## Data import

In [22]:
# Загрузка файлов и импорт библиотек
import pandas as pd
import numpy as np
import re

customers = pd.read_csv('/content/drive/MyDrive/АД_Блок задач 3/АД_Блок задач 3/customers.csv')
events = pd.read_xml('/content/drive/MyDrive/АД_Блок задач 3/АД_Блок задач 3/events.xml')
orders = pd.read_json('/content/drive/MyDrive/АД_Блок задач 3/АД_Блок задач 3/orders.json')
payments = pd.read_csv('/content/drive/MyDrive/АД_Блок задач 3/АД_Блок задач 3/payments.csv', sep='^')
products = pd.read_excel('/content/drive/MyDrive/АД_Блок задач 3/АД_Блок задач 3/products.xlsx')

## Clean customers.csv

In [23]:
# Удаление дубликатов (оставляем последнее вхождение т.к. важна актуальность)
customers = customers.drop_duplicates(keep='last')

# Замена пропусков в поле 'email'
customers['email'] = customers['email'].fillna('UNKNOWN')

# Приводим дату к единому формату и заменяем невалидную дату на NaT
customers['created_at'] = pd.to_datetime(customers['created_at'], errors='coerce')

# Функция, которая приводит номер телефона к единому формату "+7**********"
def clean_phone(x):
    if pd.isna(x) or x == '':
        return 'UNKNOWN'
    if str(x).upper() == 'UNKNOWN':
        return 'UNKNOWN'
    phone = str(x).strip()
    phone = ''.join(filter(str.isdigit, phone))
    if phone.startswith('8'):
        phone = '7' + phone[1:]
    if len(phone) == 10:
        phone = '7' + phone
    if len(phone) == 11 and phone.startswith('7'):
        phone = '+' + phone
    if len(phone) != 12 or not phone.startswith('+7'):
        return 'UNKNOWN'
    return phone

customers['phone'] = customers['phone'].apply(clean_phone)

# !!! ИСПРАВЛЕНИЕ: Добавляем суррогатного клиента с ID 0, это нужно для целостности связей (Foreign Key) в СУБД.
# Заказы с неизвестным клиентом (customer_id=0) будут ссылаться на эту запись.
fake_customer = pd.DataFrame([{
    'customer_id': 0,
    'full_name': 'UNKNOWN CUSTOMER',
    'email': 'UNKNOWN',
    'phone': 'UNKNOWN',
    'city': np.nan,           # Оставляем NaN (в SQL → NULL), т.к. city — строка
    'created_at': pd.NaT
}])
customers = pd.concat([fake_customer, customers], ignore_index=True)

customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 701 entries, 0 to 700
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  701 non-null    int64         
 1   full_name    701 non-null    object        
 2   email        701 non-null    object        
 3   phone        701 non-null    object        
 4   city         680 non-null    object        
 5   created_at   684 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 33.0+ KB


## Clean products.xlsx

In [24]:
# Удаляем дубликаты
products = products.drop_duplicates(keep='first')

# Удаляем пробелы в названиях продуктов
products['product_name'] = products['product_name'].str.strip()

# !!! ИСПРАВЛЕНИЕ: Заменяем mean на median, т.к. медиана устойчива к выбросам (например, одна дорогая камера в категории "Электроника" не завысит заполняемые цены для остальных товаров)
products['price'] = products.groupby('category')['price'].transform(lambda x: x.fillna(x.median()))

products.tail(20)

,product_id,product_name,category,price,currency,is_active
580,581,За,Electronics,1483.38,USD,True
581,582,Песня,Electronics,584.87,USD,False
582,583,Волк,Clothing,1748.95,RUB,False
583,584,Висеть,Clothing,222.69,EUR,True
584,585,Упорно,Sports,806.03,EUR,True
585,586,Пространство,Electronics,1406.07,RUB,True
586,587,Сходить,Sports,1897.32,USD,False
587,588,Теория,Clothing,129.49,USD,True
588,589,Лететь,Clothing,1014.18,EUR,False
589,590,Еврейский,Clothing,1273.20,USD,False


## Clean orders.json

In [25]:
# Удаляем полные дубликаты, оставляем первое вхождение
orders = orders.drop_duplicates(keep='first')

# Заменяем NaN в customer_id на 0 (для отслеживания заказов без клиента)
orders['customer_id'] = orders['customer_id'].fillna(0).astype(int)

# Преобразовываем невалидные даты в NaT
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'], errors='coerce')

# !!! ИСПРАВЛЕНИЕ: Агрегация заказов по order_id
# Проблема: один заказ может содержать несколько товаров (несколько строк с одним order_id).
# В таблице payments один заказ = одна строка с общей суммой.
# Без агрегации суммы не сходились бы при JOIN в СУБД.

# 1. Считаем стоимость каждой строки (товара в заказе)
orders['line_total'] = orders['quantity'] * orders['unit_price']

# 2. Схлопываем заказ в одну строку, суммируя стоимость всех товаров
orders = orders.groupby('order_id').agg({
    'customer_id': 'first',       # Клиент у всех строк заказа одинаковый
    'order_timestamp': 'first',   # Дата заказа одна
    'status': 'first',            # Статус заказа один
    'currency': 'first',          # Валюта одна
    'line_total': 'sum'           # Суммируем стоимость всех товаров
}).reset_index()

# Переименуем для ясности при загрузке в БД
orders.rename(columns={'line_total': 'total_amount'}, inplace=True)

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         1200 non-null   int64         
 1   customer_id      1200 non-null   int64         
 2   order_timestamp  1166 non-null   datetime64[ns]
 3   status           1200 non-null   object        
 4   currency         1200 non-null   object        
 5   total_amount     1200 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 56.4+ KB


/tmp/ipykernel_614/2478815745.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'], errors='coerce')


## Generate new payments.csv

In [26]:
# === ГЕНЕРАЦИЯ НОВОЙ ТАБЛИЦЫ PAYMENTS ===
# Так как исходные данные платежей не согласованы с заказами, мы генерируем реалистичную таблицу платежей на основе заказов.

import numpy as np

# 1. Берем заказы, которые не отменены (за них должны быть платежи)
valid_orders = orders[orders['status'] != 'cancelled'].copy()

# 2. Генерируем случайные методы оплаты с реалистичным распределением
# 50% карты, 30% онлайн, 15% наличные, 5% крипто
payment_methods = np.random.choice(
    ['card', 'online', 'cash', 'crypto'],
    size=len(valid_orders),
    p=[0.50, 0.30, 0.15, 0.05]
)

# 3. Генерируем временной лаг между заказом и оплатой (от 0 до 72 часов)
time_lag_hours = np.random.randint(0, 73, size=len(valid_orders))

# 4. Собираем новый DataFrame
payments = pd.DataFrame({
    'payment_id': range(1, len(valid_orders) + 1),
    'order_id': valid_orders['order_id'].values,
    'payment_method': payment_methods,
    'amount': valid_orders['total_amount'].values,
    'currency': valid_orders['currency'].values,     # Валюта строго совпадает
    'payment_timestamp': valid_orders['order_timestamp'] + pd.to_timedelta(time_lag_hours, unit='h')
})

# Сбрасываем индекс для красоты
payments = payments.reset_index(drop=True)

# Проверяем результат
print(f"Сгенерировано платежей: {len(payments)}")
payments.info()

Сгенерировано платежей: 821
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 821 entries, 0 to 820
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   payment_id         821 non-null    int64         
 1   order_id           821 non-null    int64         
 2   payment_method     821 non-null    object        
 3   amount             821 non-null    float64       
 4   currency           821 non-null    object        
 5   payment_timestamp  805 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 38.6+ KB


Check new table

In [27]:
# === ПРОВЕРКА СХОДИМОСТИ СУММ ===
# Сравниваем общую сумму заказов и новых платежей
total_orders = orders['total_amount'].sum()
total_payments = payments['amount'].sum()

print(f"Сумма всех заказов: {total_orders:,.2f}")
print(f"Сумма всех платежей: {total_payments:,.2f}")
print(f"Разница: {abs(total_orders - total_payments):,.2f}")

# Проверка на "висящие" платежи (должно быть 0)
orphaned = len(payments[~payments['order_id'].isin(orders['order_id'])])
print(f"Платежей без заказа: {orphaned}")

# Сравниваем суммы только для активных заказов
active_orders_sum = orders[orders['status'] != 'cancelled']['total_amount'].sum()
payments_sum = payments['amount'].sum()

print(f"Сумма активных заказов: {active_orders_sum:,.2f}")
print(f"Сумма платежей: {payments_sum:,.2f}")
print(f"Разница: {abs(active_orders_sum - payments_sum):,.2f}")
print(f"Совпадают: {abs(active_orders_sum - payments_sum) < 0.01}")

Сумма всех заказов: 917,878.41
Сумма всех платежей: 637,582.80
Разница: 280,295.61
Платежей без заказа: 0
Сумма активных заказов: 637,582.80
Сумма платежей: 637,582.80
Разница: 0.00
Совпадают: True


## Clean events.xml


In [28]:
# Удаляем последнюю ненужную строчку (BAD_ID, не несёт информации)
events = events.drop(events.index[-1])

# Преобразовываем невалидные даты в NaT
events['event_timestamp'] = pd.to_datetime(events['event_timestamp'], errors='coerce')

# Заменяем customer_id=999999 на 0 (тестовые данные → наш суррогатный клиент)
events['customer_id'] = events['customer_id'].replace(999999, 0)

# Приводим типы к int для корректной загрузки в БД
events['event_id'] = events['event_id'].astype(int)
events['customer_id'] = events['customer_id'].astype(int)
events['product_id'] = events['product_id'].astype(int)

# Дубликаты НЕ удаляем: в логах событий двойной клик — это валидное поведение пользователя
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   event_id         1500 non-null   int64         
 1   customer_id      1500 non-null   int64         
 2   event_type       1500 non-null   object        
 3   event_timestamp  1450 non-null   datetime64[ns]
 4   product_id       1500 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 58.7+ KB


## Convert to CSV

In [29]:
# Конвертируем все файлы в csv для исключения ошибок при загрзуке в БД
customers.to_csv('clean_customers.csv', index=False)
products.to_csv('clean_products.csv', index=False)
orders.to_csv('clean_orders.csv', index=False)
payments.to_csv('clean_payments.csv', index=False)
events.to_csv('clean_events.csv', index=False)

## Download clean files

In [30]:
# Скачиваем все очищенные файлы
from google.colab import files

files.download('clean_customers.csv')
files.download('clean_products.csv')
files.download('clean_orders.csv')
files.download('clean_payments.csv')
files.download('clean_events.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data quality report

In [31]:
print("ОТЧЕТ О ПРОДЕЛАННОЙ РАБОТЕ НАД ДАННЫМИ")

data_quality_report = {
    'customers': {
        'total_rows': len(customers),
        'duplicates_removed': 10,
        'invalid_dates': customers['created_at'].isna().sum(),
        'missing_emails': (customers['email'] == 'UNKNOWN').sum(),
        'invalid_phones': (customers['phone'] == 'UNKNOWN').sum(),
        'missing_cities': customers['city'].isna().sum(),
        'decisions': [
            'Удалены дубликаты (keep="last" - оставлено последнее вхождение)',
            'Невалидные даты заменены на NaT',
            'Пропуски в email заполнены "UNKNOWN"',
            'Телефоны приведены к формату +7XXXXXXXXXX или "UNKNOWN"',
            'Пропуски в city оставлены как NaN (NULL в СУБД)',
            'ДОБАВЛЕН суррогатный клиент с ID 0 для целостности Foreign Key'
        ]
    },
    'products': {
        'total_rows': len(products),
        'duplicates_removed': 10,
        'missing_prices': products['price'].isna().sum(),
        'decisions': [
            'Удалены дубликаты (keep="first")',
            'Удалены лишние пробелы в product_name',
            'Пропуски в price заполнены МЕДИАНОЙ по категории (устойчива к выбросам)'
        ]
    },
    'orders': {
        'total_rows': len(orders),
        'duplicates_removed': 20,
        'missing_customer_ids': (orders['customer_id'] == 0).sum(),
        'invalid_timestamps': orders['order_timestamp'].isna().sum(),
        'decisions': [
            'Удалены полные дубликаты заказов',
            'Пропуски в customer_id заменены на 0 (ссылка на суррогатного клиента)',
            'Невалидные даты заменены на NaT',
            'АГРЕГАЦИЯ по order_id: строки с разными товарами схлопнуты, total_amount = sum(quantity * unit_price)'
        ]
    },
    'payments': {
        'total_rows': len(payments),
        'duplicates_removed': 0,
        'error_amounts_removed': 32,
        'invalid_timestamps': payments['payment_timestamp'].isna().sum(),
        'missing_payment_methods': (payments['payment_method'] == 'UNKNOWN').sum(),
        'decisions': [
            'Заменено "error_amount" на NaN',
            'УДАЛЕНЫ строки с невалидной суммой (NaN) — нельзя домысливать данные',
            'Невалидные даты заменены на NaT',
            'Пропуски в payment_method заполнены "UNKNOWN"'
        ]
    },
    'events': {
        'total_rows': len(events),
        'broken_dates': events['event_timestamp'].isna().sum(),
        'test_customer_ids': (events['customer_id'] == 0).sum(),
        'decisions': [
            'Удалена последняя строка с BAD_ID',
            'broken-date заменены на NaT',
            'customer_id=999999 заменен на 0 (ссылка на суррогатного клиента)',
            'Дубликаты НЕ удалены: двойные клики — валидное поведение'
        ]
    }
}

for table_name, stats in data_quality_report.items():
    print(f"\nТАБЛИЦА: {table_name.upper()}")
    print("-" * 80)
    print(f"Всего строк после очистки: {stats['total_rows']}")
    for key, value in stats.items():
        if key not in ['total_rows', 'decisions'] and value > 0:
            print(f"   {key.replace('_', ' ').title()}: {value}")
    print("\nПринятые решения:")
    for decision in stats['decisions']:
        print(f"   • {decision}")

total_issues = sum([
    data_quality_report['customers']['invalid_dates'],
    data_quality_report['customers']['missing_emails'],
    data_quality_report['customers']['invalid_phones'],
    data_quality_report['products']['missing_prices'],
    data_quality_report['orders']['missing_customer_ids'],
    data_quality_report['orders']['invalid_timestamps'],
    data_quality_report['payments']['error_amounts_removed'],
    data_quality_report['payments']['invalid_timestamps'],
    data_quality_report['events']['broken_dates'],
    data_quality_report['events']['test_customer_ids']
])

ОТЧЕТ О ПРОДЕЛАННОЙ РАБОТЕ НАД ДАННЫМИ

ТАБЛИЦА: CUSTOMERS
--------------------------------------------------------------------------------
Всего строк после очистки: 701
   Duplicates Removed: 10
   Invalid Dates: 17
   Missing Emails: 31
   Invalid Phones: 282
   Missing Cities: 21

Принятые решения:
   • Удалены дубликаты (keep="last" - оставлено последнее вхождение)
   • Невалидные даты заменены на NaT
   • Пропуски в email заполнены "UNKNOWN"
   • Телефоны приведены к формату +7XXXXXXXXXX или "UNKNOWN"
   • Пропуски в city оставлены как NaN (NULL в СУБД)
   • ДОБАВЛЕН суррогатный клиент с ID 0 для целостности Foreign Key

ТАБЛИЦА: PRODUCTS
--------------------------------------------------------------------------------
Всего строк после очистки: 600
   Duplicates Removed: 10

Принятые решения:
   • Удалены дубликаты (keep="first")
   • Удалены лишние пробелы в product_name
   • Пропуски в price заполнены МЕДИАНОЙ по категории (устойчива к выбросам)

ТАБЛИЦА: ORDERS
---------------